### ENV를 통해 OPEN AI API 연결


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPEN_API_KEY")

client = OpenAI(api_key=api_key)

print(client.api_key)

### OPEN AI 연결 후 첫 질문과 답변


In [ ]:
from openai import OpenAI
import openai
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

completion = client.chat.completions.create(
    model="gpt-4o-mini", messages=[{"role": "user", "content": "Hello World"}]
)

print("completion")
print()
print("completion.choices : ", completion.choices[0].message.content)
print()
print("completion.usage.completion_tokens : ", completion.usage.completion_tokens)
print()
print("completion.usage.total_tokens : ", completion.usage.total_tokens)

### OPEN AI CLIENT 만들기


In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
        {"role": "user", "content": "파인튜닝이 무엇인지 한 문장으로 설명해주세요."},
    ],
    temperature=0.7,
    max_tokens=256,
    stream=False,
    n=2,
)

# print(response.choices[0].message.content)
# print()
# print(response.choices[1].message.content)

for cs in response.choices:
    print(cs.message.content)
    print()

### CLIENT 생성 시 SYSTEM 설정의 차이


In [ ]:
response_no_sys = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "계약 해지 통보는 언제 해야 하나요?"}],
    temperature=0.3,
    max_tokens=200,
)

response_with_sys = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "당신은 대한민국 법률 전문가입니다. 관련 법조항을 인용하여 3문장 이내로 답변하세요.",
        },
        {"role": "user", "content": "계약 해지 통보는 언제 해야 되나요?"},
    ],
    temperature=0.3,
    max_tokens=200,
)


print("===SYSTEM -X-===")
print(response_no_sys.choices[0].message.content)

print("===SYSTEM -O-===")
print(response_with_sys.choices[0].message.content)

### RESPONSE 구조 파헤치기


In [ ]:
print("답변 내용 : ", response.choices[0].message.content)

print()

print("종료 이유 : ", response.choices[0].finish_reason)

print()

print("입력 토큰 : ", response.usage.prompt_tokens)

print()

print("출력 토큰 : ", response.usage.completion_tokens)

print()

print("전체 토큰 : ", response.usage.total_tokens)

### MAX_TOKEN을 너무 작게 설정할 시 상황


In [ ]:
response_short = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "대한민국 헌법의 기본 원리를 상세히 설명해 주세요."}
    ],
    max_tokens=20,
)

print("답변 : ", response_short.choices[0].message.content)
print("종료 이유 : ", response_short.choices[0].finish_reason)
print("-> 문장이 중간에 잘렸습니다! max_tokens를 늘려야 합니다.")

### 비용 개산


In [ ]:
INPUT_PRICE = 0.15
OUTPUT_PRICE = 0.60

input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens

total_cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000

print(f"입력 : {input_tokens}토큰 / 출력 : {output_tokens}")
print()
print(f"이번 호출 비용 : ${total_cost:.6f} (약 {total_cost * 1400:.4f}원)")
print(f"1,000번 호출하면: 약 {total_cost * 1000 * 1400:.0f}원")

### 누적 비용 추적기


In [ ]:
class TokenTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.call_count = 0

    def add(self, response):
        self.total_input += response.usage.prompt_tokens
        self.total_output += response.usage.completion_tokens
        self.call_count += 1

    def report(self):
        cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(
            f"총 {self.call_count}회 호출 | 입력 {self.total_input} + 출력 {self.total_output} 토큰"
        )
        print(f"누적 비용: ${cost:.4f} (약 {cost * 1400:.0f}원)")


tracker = TokenTracker()
tracker.add(response)
tracker.report()